# XGBoost walk-forward baseline

Vinit Bhatt · Mar 2026

Loads ETF vol CSV → builds features → walk-forward XGBoost per ticker → saves `baseline_xgb_walkforward`.
Set `SELECTED_TICKER = 'SPY'` to run one ticker, `None` for all.


In [ ]:
!pip -q install -U xgboost scikit-learn
print('done')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from __future__ import annotations
import os, math, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

In [ ]:
DRIVE_FOLDER = '/content/drive/MyDrive/Colab Notebooks/'
INPUT_CSV    = 'vol_dataset_with_4_baselines_022326.csv'  # update if needed

def find_csv(folder):
    try:
        files = sorted(f for f in os.listdir(folder) if f.endswith('.csv'))
        return os.path.join(folder, files[0]) if files else None
    except FileNotFoundError:
        return None

DATA_PATH = os.path.join(DRIVE_FOLDER, INPUT_CSV) if INPUT_CSV else find_csv(DRIVE_FOLDER)
print('file:', DATA_PATH)

DATE_COL   = 'date'
TICKER_COL = 'ticker'
TARGET_COL = 'forward_vol_5d_annual_decimel_calculated'
BASELINE_COLS = ['baseline_avg_5', 'baseline_avg_20', 'baseline_ols_5', 'baseline_ols_20']

SELECTED_TICKER      = None   # 'SPY' for one ticker
OUTPUT_ONLY_SELECTED = True
USE_BASELINES_AS_FEATURES = False  # True = stacking

MAX_LAG   = 20
MIN_TRAIN = 125   # ~6 months of rows
STEP      = 252   # retrain every ~1yr
ES_SPLIT  = 0.85
ES_ROUNDS = 50

XGB_PARAMS = dict(
    n_estimators=2000,   # early stopping cuts this
    learning_rate=0.03, max_depth=3,
    subsample=0.8, colsample_bytree=0.8,
    reg_lambda=1.0, min_child_weight=1.0,
    objective='reg:squarederror', eval_metric='rmse',
    random_state=42, n_jobs=-1,
)

In [ ]:
assert DATA_PATH and os.path.exists(DATA_PATH), f'file not found: {DATA_PATH}'

df_raw = pd.read_csv(DATA_PATH, low_memory=False)

missing = {DATE_COL, TICKER_COL, TARGET_COL} - set(df_raw.columns)
assert not missing, f'missing cols: {missing}'

df = df_raw.copy()
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values([TICKER_COL, DATE_COL]).reset_index(drop=True)

print(df.shape, '|', df[TICKER_COL].nunique(), 'tickers')
print(df[DATE_COL].min().date(), '->', df[DATE_COL].max().date())
print('target fill:', round(df[TARGET_COL].notna().mean(), 3))

all_tickers    = sorted(df[TICKER_COL].dropna().unique())
tickers_to_run = [SELECTED_TICKER] if SELECTED_TICKER else all_tickers
print('running:', len(tickers_to_run), 'tickers')

In [ ]:
# features — only info known at time t, lags via groupby.shift so no look-ahead
FEAT_CANDIDATES = [
    'y_known_at_t',   # lagged target = persistence signal
    'trailing_vol_annual_decimel_5d_factset',
    'trailing_vol_annual_decimel_20d_factset',
    'trailing_vol__annual_decimel_5d_calculated',
    'trailing_vol__annual_decimel_20d_calculated',
    'trailing_vol_annual_decimel_5d_calculated',
    'trailing_vol_annual_decimel_20d_calculated',
    'VIX', 'NYGOLDS', 'OIL_WTI_S', 'US_10Y_BOND_YLD', 'US_3M_TB_YLD',
]
if USE_BASELINES_AS_FEATURES:
    FEAT_CANDIDATES = ['baseline_avg_5','baseline_avg_20',
                       'baseline_ols_5','baseline_ols_20'] + FEAT_CANDIDATES

base_feats = [c for c in FEAT_CANDIDATES if c in df.columns]

df_feat = df.copy()
df_feat['dow']     = df_feat[DATE_COL].dt.dayofweek
df_feat['month']   = df_feat[DATE_COL].dt.month
df_feat['quarter'] = df_feat[DATE_COL].dt.quarter
cal_feats = ['dow', 'month', 'quarter']

lag_src = [c for c in ['y_known_at_t',
    'trailing_vol__annual_decimel_5d_calculated',
    'trailing_vol__annual_decimel_20d_calculated',
    'trailing_vol_annual_decimel_5d_calculated',
    'trailing_vol_annual_decimel_20d_calculated',
    'VIX'] if c in df_feat.columns]

lag_feats = []
for col in lag_src:
    for L in [1, 5, 20]:
        nm = f'{col}_lag{L}'
        df_feat[nm] = df_feat.groupby(TICKER_COL)[col].shift(L)
        lag_feats.append(nm)

FEATURE_COLS = list(dict.fromkeys(base_feats + cal_feats + lag_feats))
print(f'{len(FEATURE_COLS)} features  |  base: {base_feats}')

In [ ]:
def warmup_start(g, tcol, max_lag, min_rows):
    """Walk forward until we have enough lag history + enough training rows."""
    n_target = 0
    for i, val in enumerate(g[tcol]):
        if i >= max_lag and n_target >= min_rows:
            return i
        if pd.notna(val):
            n_target += 1
    return None


def run_walkforward(df_full, tickers, feat_cols, target_col, date_col,
                    ticker_col, max_lag, min_train, step, es_split, params):
    preds = pd.Series(np.nan, index=df_full.index, name='baseline_xgb_walkforward')
    log   = []

    for tkr in tickers:
        g = df_full[df_full[ticker_col] == tkr].sort_values(date_col).copy()
        if g.empty:
            continue

        t0 = warmup_start(g, target_col, max_lag, min_train)
        if t0 is None:
            print(f'{tkr}: skipped (never warmed up)')
            log.append({'ticker': tkr, 'status': 'skipped', 'rows': len(g)})
            continue

        print(f'{tkr}: {len(g)} rows, first pred {g.iloc[t0][date_col].date()}')

        for bs in range(t0, len(g), step):
            be    = min(bs + step, len(g))
            train = g.iloc[:bs].copy()
            pred  = g.iloc[bs:be].copy()

            if not train.empty:
                assert train[date_col].max() < pred[date_col].min(), 'date overlap!'

            tr = train.dropna(subset=[target_col])
            if len(tr) < min_train:
                continue

            sp         = max(1, min(int(len(tr) * es_split), len(tr) - 1))
            Xtr, ytr   = tr.iloc[:sp][feat_cols], tr.iloc[:sp][target_col]
            Xval, yval = tr.iloc[sp:][feat_cols], tr.iloc[sp:][target_col]

            m     = fit_xgb(Xtr, ytr, Xval, yval, params)
            y_hat = np.clip(m.predict(pred[feat_cols]), 0, None)
            preds.loc[pred.index] = y_hat

            sc = pred.dropna(subset=[target_col])
            if len(sc):
                r = metrics(sc[target_col], preds.loc[sc.index])
                rmse, mae, r2 = r['RMSE'], r['MAE'], r['R2']
            else:
                rmse = mae = r2 = float('nan')

            log.append({'ticker': tkr, 'status': 'ok',
                        'block_start': str(pred[date_col].min().date()),
                        'block_end':   str(pred[date_col].max().date()),
                        'train_end':   str(train[date_col].max().date()) if not train.empty else None,
                        'n_train': len(tr), 'n_pred': len(pred), 'n_scored': len(sc),
                        'rmse': rmse, 'mae': mae, 'r2': r2,
                        'best_iter': int(getattr(m, 'best_iteration', 0) or 0)})

        print(f'  {preds.loc[g.index].notna().mean():.0%} coverage')

    print(f'done — overall fill: {preds.notna().mean():.1%}')
    return preds, pd.DataFrame(log)


In [ ]:
preds, diag_df = run_walkforward(
    df_feat, tickers_to_run, FEATURE_COLS,
    TARGET_COL, DATE_COL, TICKER_COL,
    MAX_LAG, MIN_TRAIN, STEP, ES_SPLIT, XGB_PARAMS
)

df_out = df_feat.copy()
df_out['baseline_xgb_walkforward'] = preds.values

display(diag_df.head(20))

In [ ]:
compare_cols = BASELINE_COLS + ['baseline_xgb_walkforward', TARGET_COL]
compare_cols = [c for c in compare_cols if c in df_out.columns]

ev = df_out.dropna(subset=compare_cols).copy()
print(f'{len(ev):,} rows in eval')

rows  = [metrics(ev[TARGET_COL], ev[c], c) for c in BASELINE_COLS if c in ev]
rows += [metrics(ev[TARGET_COL], ev['baseline_xgb_walkforward'], 'xgb_walkforward')]
display(pd.DataFrame(rows).set_index('Model').sort_values('RMSE').round(5))

In [ ]:
if SELECTED_TICKER and OUTPUT_ONLY_SELECTED:
    out_df   = df_out[df_out[TICKER_COL] == SELECTED_TICKER].copy()
    out_name = f'vol_{SELECTED_TICKER}_xgb_wf.csv'
else:
    out_df   = df_out.copy()
    out_name = 'vol_all_xgb_wf.csv'

out_df.to_csv(os.path.join(DRIVE_FOLDER, out_name), index=False)
diag_df.to_csv(os.path.join(DRIVE_FOLDER, 'xgb_wf_diag.csv'), index=False)
print('saved:', out_name)

In [ ]:
col = df_out['baseline_xgb_walkforward']
print('negatives:', (col.dropna() < 0).sum(), ' (want 0)')
print(f'range {col.min():.4f} – {col.max():.4f}  mean {col.mean():.4f}')

# nan rate by year — high early, should hit 0
tmp = df_out[[DATE_COL, TICKER_COL, 'baseline_xgb_walkforward']].copy()
tmp['year'] = tmp[DATE_COL].dt.year
display(tmp.groupby('year')['baseline_xgb_walkforward']
           .apply(lambda s: s.isna().mean())
           .to_frame('nan_rate').round(3))

# no gaps after first prediction per ticker
def has_gap(s):
    first = s.first_valid_index()
    return s.loc[first:].isna().any() if first else True

gaps = df_out.groupby(TICKER_COL)['baseline_xgb_walkforward'].apply(has_gap)
print('tickers with gaps:', gaps.sum())
if gaps.any(): print(gaps[gaps].index.tolist())